
# 🚀 Crypto Trading — XRP (CMC ➜ Binance) — **v3 com timeframe 1H completo**

**Novidades desta versão:**
- Além dos gráficos existentes (4H 3m, 1H 24h, MACD/RSI 3D/1D/8H, EMA21 4H), agora inclui:
  - **1H (janela longa)** para análise tática;
  - **MACD 1H**, **RSI 1H** e **EMA21 1H** com sinais (Buy/Sell, Overbought/Oversold, Entry/Exit).


In [ ]:

# 1) Imports, config e utilidades
import os, json
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas_ta as ta
    HAS_PTA = True
except Exception:
    HAS_PTA = False

SYMBOL = "XRP"
FIAT = "USD"
LOCAL_TZ = "America/Sao_Paulo"
CMC_API_KEY = os.getenv("CMC_API_KEY")

def to_local_tz(df, tz=LOCAL_TZ):
    if df is None or df.empty: return df
    df.index = pd.to_datetime(df.index, utc=True)
    return df.tz_convert(tz)

def resample_ohlc(df, rule):
    rule = str(rule).lower()
    o = df["Open"].resample(rule).first()
    h = df["High"].resample(rule).max()
    l = df["Low"].resample(rule).min()
    c = df["Close"].resample(rule).last()
    v = df["Volume"].resample(rule).sum()
    out = pd.concat([o,h,l,c,v], axis=1).dropna()
    out.columns = ["Open","High","Low","Close","Volume"]
    return out

def ema(s, span): return s.ewm(span=span, adjust=False).mean()

def macd_series(close, fast=12, slow=26, sig=9):
    if HAS_PTA:
        m = ta.macd(close, fast=fast, slow=slow, signal=sig).dropna()
        return m[f"MACD_{fast}_{slow}_{sig}"], m[f"MACDs_{fast}_{slow}_{sig}"], m[f"MACDh_{fast}_{slow}_{sig}"]
    macd_line = ema(close, fast) - ema(close, slow)
    signal = ema(macd_line, sig)
    hist = macd_line - signal
    return macd_line.dropna(), signal.dropna(), hist.dropna()

def rsi_series(close, length=14):
    if HAS_PTA:
        return ta.rsi(close, length=length).dropna()
    delta = close.diff()
    gain = (delta.clip(lower=0)).ewm(alpha=1/length, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/length, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.dropna()

def macd_signals(close):
    macd_line, signal, hist = macd_series(close)
    cross = np.sign(macd_line - signal).diff()
    last_cross_idx = cross.dropna().ne(0).iloc[::-1].idxmax() if not cross.dropna().empty else None
    last_signal = None
    if last_cross_idx is not None and last_cross_idx in cross.index:
        last_signal = "BUY" if cross.loc[last_cross_idx] > 0 else "SELL"
    bias = "BULLISH" if macd_line.iloc[-1] > signal.iloc[-1] else "BEARISH"
    return macd_line, signal, hist, last_signal, bias

def rsi_state(close):
    rsi = rsi_series(close)
    st = "NEUTRAL"
    if rsi.iloc[-1] >= 70: st = "OVERBOUGHT"
    elif rsi.iloc[-1] <= 30: st = "OVERSOLD"
    return rsi, st


In [ ]:

# 2) CMC silencioso + Binance fallback
CMC_BASE = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/ohlcv/historical"
BINANCE_BASE = "https://api.binance.com/api/v3/klines"
BINANCE_SYMBOL = "XRPUSDT"

def cmc_ohlcv(symbol, start=None, end=None, interval="1d", convert=FIAT):
    params = {"symbol": symbol, "convert": convert, "interval": interval}
    if start: params["time_start"] = start
    if end:   params["time_end"]   = end
    headers = {"X-CMC_PRO_API_KEY": CMC_API_KEY} if CMC_API_KEY else {}
    try:
        r = requests.get(CMC_BASE, params=params, headers=headers, timeout=60)
        if r.status_code != 200:
            return None
        data = r.json()
        quotes = data["data"]["quotes"]
        rows = []
        for q in quotes:
            t = q["time_open"]
            o = q["quote"][convert]["open"]
            h = q["quote"][convert]["high"]
            l = q["quote"][convert]["low"]
            c = q["quote"][convert]["close"]
            v = q["quote"][convert]["volume"]
            rows.append([t,o,h,l,c,v])
        df = pd.DataFrame(rows, columns=["Datetime","Open","High","Low","Close","Volume"])
        df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True)
        df = df.set_index("Datetime").sort_index()
        return to_local_tz(df)
    except Exception:
        return None

def binance_klines(symbol, interval, limit=1500):
    r = requests.get(BINANCE_BASE, params={"symbol":symbol, "interval":interval, "limit":limit}, timeout=60)
    r.raise_for_status()
    data = r.json()
    rows = []
    for k in data:
        open_time = pd.to_datetime(k[0], unit="ms", utc=True)
        o,h,l,c,v = map(float, (k[1],k[2],k[3],k[4],k[5]))
        rows.append([open_time,o,h,l,c,float(v)])
    df = pd.DataFrame(rows, columns=["Datetime","Open","High","Low","Close","Volume"]).set_index("Datetime").sort_index()
    return to_local_tz(df)


In [ ]:

# 3) Coleta de dados (inclui 1H longa)
from datetime import timezone

def get_4h_3m():
    now = datetime.now(timezone.utc)
    start = (now - timedelta(days=95)).strftime("%Y-%m-%dT%H:%M:%SZ")
    df = cmc_ohlcv(SYMBOL, start=start, interval="4h")
    if df is None:
        df1h = cmc_ohlcv(SYMBOL, start=start, interval="1h")
        if df1h is not None:
            df = resample_ohlc(df1h, "4h")
    if df is None:
        df = binance_klines(BINANCE_SYMBOL, "4h", limit=1000)
    return df.loc[df.index.max()-pd.Timedelta(days=90):]

def get_1h_24h():
    now = datetime.now(timezone.utc)
    start = now - timedelta(hours=30)
    df = cmc_ohlcv(SYMBOL, start=start.strftime("%Y-%m-%dT%H:%M:%SZ"), interval="1h")
    if df is None:
        df = binance_klines(BINANCE_SYMBOL, "1h", limit=200)
    return df.loc[df.index.max()-pd.Timedelta(hours=24):]

def get_1h_long(days=30, limit_binance=1500):
    # Janela mais longa de 1H para indicadores (padrão 30 dias)
    now = datetime.now(timezone.utc)
    start = (now - timedelta(days=days+2)).strftime("%Y-%m-%dT%H:%M:%SZ")
    df = cmc_ohlcv(SYMBOL, start=start, interval="1h")
    if df is None:
        df = binance_klines(BINANCE_SYMBOL, "1h", limit=limit_binance)
    return df.loc[df.index.max()-pd.Timedelta(days=days):]

def get_1d_200d():
    now = datetime.now(timezone.utc)
    start = (now - timedelta(days=220)).strftime("%Y-%m-%dT%H:%M:%SZ")
    df = cmc_ohlcv(SYMBOL, start=start, interval="1d")
    if df is None:
        df = binance_klines(BINANCE_SYMBOL, "1d", limit=500)
    return df

def get_8h_10d():
    df1h = get_1h_long(days=10, limit_binance=500)  # reutiliza 1h longa
    return resample_ohlc(df1h, "8h")

# Baixa tudo
df_4h_3m = get_4h_3m()
df_1h_24h = get_1h_24h()
df_1h_long = get_1h_long(days=30)  # 30 dias padrão
df_1d = get_1d_200d()
df_3d = resample_ohlc(df_1d, "3d")
df_8h = get_8h_10d()

print("Amostras -> 4H(3m):", len(df_4h_3m), "| 1H(24h):", len(df_1h_24h), "| 1H(longo):", len(df_1h_long), 
      "| 1D:", len(df_1d), "| 3D:", len(df_3d), "| 8H:", len(df_8h))


In [ ]:

# 4) Plot helpers (candle com fallback)
def plot_candles(df, title):
    try:
        import mplfinance as mpf
        mpf.plot(df, type="candle", volume=True, title=title, style="classic")
    except Exception:
        ax = df["Close"].plot(title=title, figsize=(10,4)); ax.set_xlabel("Tempo"); ax.set_ylabel(f"Preço ({FIAT})"); plt.show()

def plot_lines(series_dict, title):
    plt.figure()
    for label, s in series_dict.items():
        s.dropna().plot(label=label)
    plt.title(title); plt.legend(); plt.xlabel("Tempo"); plt.ylabel("Valor"); plt.show()


In [ ]:

# 5) Candles
plot_candles(df_4h_3m, "XRP/USD — Candles 4H (3m)")
plot_candles(df_1h_24h, "XRP/USD — Candles 1H (24h)")
plot_candles(df_1h_long, "XRP/USD — Candles 1H (30d)")


In [ ]:

# 6) MACD — 3D, 1D, 8H, **1H**
m3, s3, h3, sig3, bias3 = macd_signals(df_3d["Close"]);  plot_lines({"MACD 3D": m3, "Signal 3D": s3, "Hist 3D": h3}, "XRP/USD — MACD 3D");  print("MACD 3D:", sig3, "|", bias3)
m1, s1, h1, sig1, bias1 = macd_signals(df_1d["Close"]);  plot_lines({"MACD 1D": m1, "Signal 1D": s1, "Hist 1D": h1}, "XRP/USD — MACD 1D");  print("MACD 1D:", sig1, "|", bias1)
m8, s8, h8, sig8, bias8 = macd_signals(df_8h["Close"]);  plot_lines({"MACD 8H": m8, "Signal 8H": s8, "Hist 8H": h8}, "XRP/USD — MACD 8H");  print("MACD 8H:", sig8, "|", bias8)
mH, sH, hH, sigH, biasH = macd_signals(df_1h_long["Close"]);  plot_lines({"MACD 1H": mH, "Signal 1H": sH, "Hist 1H": hH}, "XRP/USD — MACD 1H");  print("MACD 1H:", sigH, "|", biasH)


In [ ]:

# 7) RSI — 3D, 1D, 8H, **1H**
r3, st3 = rsi_state(df_3d["Close"]);  plot_lines({"RSI 3D": r3}, "XRP/USD — RSI 3D");  print("RSI 3D:", st3)
r1, st1 = rsi_state(df_1d["Close"]);  plot_lines({"RSI 1D": r1}, "XRP/USD — RSI 1D");  print("RSI 1D:", st1)
r8, st8 = rsi_state(df_8h["Close"]);  plot_lines({"RSI 8H": r8}, "XRP/USD — RSI 8H");  print("RSI 8H:", st8)
rH, stH = rsi_state(df_1h_long["Close"]);  plot_lines({"RSI 1H": rH}, "XRP/USD — RSI 1H");  print("RSI 1H:", stH)


In [ ]:

# 8) EMA21 — 4H (3m) e **1H (30d)** com entry/exit
def plot_ema_entry_exit(df, title):
    df = df.copy()
    df["EMA21"] = df["Close"].ewm(span=21, adjust=False).mean()
    cross_up   = (df["Close"] > df["EMA21"]) & (df["Close"].shift(1) <= df["EMA21"].shift(1))
    cross_down = (df["Close"] < df["EMA21"]) & (df["Close"].shift(1) >= df["EMA21"].shift(1))
    try:
        import mplfinance as mpf
        fig, ax = mpf.plot(df, type="candle", volume=False, returnfig=True, title=title, style="classic")
        ax.plot(df.index, df["EMA21"])
        ax.scatter(df.index[cross_up], df["Close"][cross_up])
        ax.scatter(df.index[cross_down], df["Close"][cross_down])
        fig.show()
    except Exception:
        ax = df[["Close","EMA21"]].plot(title=title, figsize=(10,4))
        ax.scatter(df.index[cross_up], df["Close"][cross_up])
        ax.scatter(df.index[cross_down], df["Close"][cross_down])
        plt.show()
    if cross_up.iloc[-1]:   print("Último evento: ENTRY (bullish cross).")
    elif cross_down.iloc[-1]: print("Último evento: EXIT (bearish cross).")
    else: print("Último evento: Sem cruzamento na última vela.")

plot_ema_entry_exit(df_4h_3m, "XRP/USD — EMA 21 (4H, 3m)")
plot_ema_entry_exit(df_1h_long, "XRP/USD — EMA 21 (1H, 30d)")
